# Predictive Modeling Using Machine Learning

**Internship Task:** Predictive Modeling Using Machine Learning  
**Problem Type:** Binary Classification

## Objective
Build supervised machine-learning models to predict whether a student will be classified as **high performance (1)** or **lower performance (0)** from demographic, academic, and experience-related features.


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, roc_auc_score
)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
print("Libraries imported successfully.")


## 2. Load and Inspect the Dataset

In [ ]:
df = pd.read_csv("predictive_modeling_dataset.csv")
print(f"Dataset shape: {df.shape}")
display(df.head())


In [ ]:
print("Data information:")
df.info()

print("\nMissing values:")
display(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())


## 3. Clean the Data
Missing numerical values will be handled by the machine-learning pipeline using median imputation. Duplicate rows are removed before modeling.


In [ ]:
df = df.drop_duplicates().reset_index(drop=True)

print("Dataset shape after duplicate removal:", df.shape)
print("Remaining missing values:")
display(df.isnull().sum())


## 4. Exploratory Target Visualization

In [ ]:
plt.figure(figsize=(7, 5))
sns.countplot(data=df, x="Performance")
plt.title("Target Class Distribution")
plt.xlabel("Performance Class (0 = Lower, 1 = High)")
plt.ylabel("Number of Records")
plt.show()


## 5. Prepare Features and Target

In [ ]:
X = df.drop(columns=["Performance"])
y = df["Performance"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))


## 6. Train Three Supervised-Learning Models
The project compares Logistic Regression, Decision Tree, and Random Forest. Imputation is included in every pipeline so missing values are handled without leaking information from the test set into training.


In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, random_state=42))
    ]),
    "Decision Tree": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", DecisionTreeClassifier(max_depth=5, random_state=42))
    ]),
    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(n_estimators=200, random_state=42))
    ])
}

results = []
trained_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1 Score": f1_score(y_test, pred)
    })
    trained_models[name] = model

results_df = pd.DataFrame(results).sort_values("Accuracy", ascending=False)
display(results_df.style.format({
    "Accuracy": "{:.3f}",
    "Precision": "{:.3f}",
    "Recall": "{:.3f}",
    "F1 Score": "{:.3f}"
}))


## 7. Accuracy Comparison

In [ ]:
plt.figure(figsize=(9, 5))
sns.barplot(data=results_df, x="Model", y="Accuracy")
plt.ylim(0, 1.05)
plt.title("Model Accuracy Comparison")
plt.ylabel("Accuracy")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


## 8. Classification Reports

In [ ]:
for name, model in trained_models.items():
    pred = model.predict(X_test)
    print("=" * 65)
    print(name)
    print("=" * 65)
    print(classification_report(
        y_test, pred,
        target_names=["Lower Performance", "High Performance"]
    ))


## 9. Confusion Matrices

In [ ]:
for name, model in trained_models.items():
    pred = model.predict(X_test)
    cm = confusion_matrix(y_test, pred)

    plt.figure(figsize=(5.5, 4.5))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=["Lower", "High"],
        yticklabels=["Lower", "High"]
    )
    plt.title(f"Confusion Matrix - {name}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.show()


## 10. ROC Curves and AUC

In [ ]:
plt.figure(figsize=(9, 6))

for name, model in trained_models.items():
    probability = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, probability)
    auc_value = roc_auc_score(y_test, probability)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc_value:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", label="Random Classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()
plt.tight_layout()
plt.show()


## 11. Select the Best Model

In [ ]:
best_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_name]
best_pred = best_model.predict(X_test)
best_prob = best_model.predict_proba(X_test)[:, 1]

print("Best model by test accuracy:", best_name)
print(f"Accuracy : {accuracy_score(y_test, best_pred):.3f}")
print(f"Precision: {precision_score(y_test, best_pred):.3f}")
print(f"Recall   : {recall_score(y_test, best_pred):.3f}")
print(f"F1 Score : {f1_score(y_test, best_pred):.3f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, best_prob):.3f}")


## 12. Key Findings and Conclusion

- The raw dataset was inspected for missing values and duplicate rows.
- Duplicate records were removed before modeling.
- Missing numerical values are handled safely inside the model pipelines using median imputation.
- Three supervised-learning algorithms were trained and compared.
- Accuracy, precision, recall, F1 score, confusion matrices, and ROC-AUC were used for evaluation.
- The model with the strongest test accuracy is identified as the best model for this experiment.

### Conclusion
This project demonstrates an end-to-end supervised machine-learning workflow: data preparation, train/test splitting, model training, performance evaluation, and visual analysis. Comparing multiple algorithms provides a practical basis for selecting a predictive model.
